In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import gc
import time
import networkx as nx
from sklearn.preprocessing import MultiLabelBinarizer
from collections import defaultdict
import textwrap

import psutil
from datetime import datetime

import anndata as an
import scanpy as sc
import owlready2 as owl

sc.settings.verbosity = 3

# Load the Data

In [2]:
dpath = "/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/"

file_list = glob.glob(f"{dpath}hsc_anndata/*.h5ad")
file_list

['/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/fibroblasts.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/mesenchymal_cells.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/myeloid_cells.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/lymphoid_cells.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/hematopoietic_progenitors.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/innate_lymphoid_cells.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/hsc.h5ad',
 '/nfs/turbo/umms-indikar/shared/projects/czi_projects/data/hsc_anndata/endothelial_cells.h5ad']

In [3]:
overall_start_time = time.time() # Start overall timer

obs_list = []

for fpath in file_list:
    loop_start_time = time.time() # Start timer for this file iteration

    basename = os.path.basename(fpath).replace(".h5ad", "")
    print(f"Working {basename}...")

    read_start_time = time.time()
    adata = sc.read_h5ad(fpath, backed='r')
    obs = adata.obs.copy()
    read_end_time = time.time()
    print(f"\tRead+Copy took: {read_end_time - read_start_time:.2f} seconds")

    obs['basename'] = basename
    print(f"\t{obs.shape=}")
    obs_list.append(obs)

    loop_end_time = time.time() # End timer for this file iteration
    print(f"\tIteration for {basename} took: {loop_end_time - loop_start_time:.2f} seconds")
    print("-" * 20) # Separator


print("Concatenating DataFrames...")
concat_start_time = time.time()
df = pd.concat(obs_list)
concat_end_time = time.time()
print(f"Concatenation took: {concat_end_time - concat_start_time:.2f} seconds")

# clean up
del adata
del obs_list
gc.collect()

print(f"{df.shape=}")

overall_end_time = time.time() # End overall timer
print(f"\nTotal script execution time: {overall_end_time - overall_start_time:.2f} seconds")

Working fibroblasts...
	Read+Copy took: 2.57 seconds
	obs.shape=(1277902, 29)
	Iteration for fibroblasts took: 2.58 seconds
--------------------
Working mesenchymal_cells...
	Read+Copy took: 1.25 seconds
	obs.shape=(650601, 29)
	Iteration for mesenchymal_cells took: 1.25 seconds
--------------------
Working myeloid_cells...
	Read+Copy took: 2.73 seconds
	obs.shape=(1147059, 29)
	Iteration for myeloid_cells took: 2.74 seconds
--------------------
Working lymphoid_cells...
	Read+Copy took: 8.79 seconds
	obs.shape=(3761959, 29)
	Iteration for lymphoid_cells took: 8.80 seconds
--------------------
Working hematopoietic_progenitors...
	Read+Copy took: 0.27 seconds
	obs.shape=(66394, 29)
	Iteration for hematopoietic_progenitors took: 0.27 seconds
--------------------
Working innate_lymphoid_cells...
	Read+Copy took: 0.19 seconds
	obs.shape=(15277, 29)
	Iteration for innate_lymphoid_cells took: 0.19 seconds
--------------------
Working hsc...
	Read+Copy took: 0.26 seconds
	obs.shape=(43544, 2

In [4]:
df.columns

Index(['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id',
       'cell_type', 'cell_type_ontology_term_id', 'development_stage',
       'development_stage_ontology_term_id', 'disease',
       'disease_ontology_term_id', 'donor_id', 'is_primary_data',
       'observation_joinid', 'self_reported_ethnicity',
       'self_reported_ethnicity_ontology_term_id', 'sex',
       'sex_ontology_term_id', 'suspension_type', 'tissue',
       'tissue_ontology_term_id', 'tissue_type', 'tissue_general',
       'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz',
       'raw_variance_nnz', 'n_measured_vars', 'basename'],
      dtype='object')

# Get a list of ontology labels

In [5]:
pdf = df.copy()
pdf = pdf[['cell_type_ontology_term_id']].drop_duplicates()
pdf = pdf.reset_index(drop=True)
print(f"{pdf.shape=}")
pdf.head()

pdf.shape=(189, 1)


,cell_type_ontology_term_id
0,CL:0000057
1,CL:0002553
2,CL:0002620
3,CL:0000186
4,CL:0002241


# Load the Ontology

In [6]:
fpath = "../resources/cl.owl"
ontology = owl.get_ontology(f"file://{fpath}").load()
base_iri = ontology.base_iri
print(f"{base_iri=}")

base_iri='http://purl.obolibrary.org/obo/cl.owl#'


In [7]:
search_term = "CL_0000000"
result = ontology.search_one(iri=f"*{search_term}")

print(f"{result=}")
print(f"{result.label=}")

result=obo.CL_0000000
result.label=['cell']


# search all classes

In [8]:
def find_lineage(term, lineage_set):
    """
    Recursively finds 'is_a' parent labels
    and adds them to the lineage_set.
    """
    # Directly access 'is_a', assuming it exists
    parents = term.is_a

    for parent in parents:
        # only interest in class hierarchy
        if not isinstance(parent, owl.ThingClass):
            continue

        # filter for cell ontology or gene ontology
        if not "CL" in parent.iri or "GO" in parent.iri:
            continue

        # Directly get label, assuming it exists and is not empty
        parent_label = parent.label.first()
        
        if parent_label not in lineage_set:
            lineage_set.add(parent_label)
            # Recurse
            find_lineage(parent, lineage_set)


def find_single_path_lineage(term, lineage_set):
    """
    Recursively finds 'is_a' parent labels along a single path,
    following only the *first* valid parent at each step,
    and adds them to the lineage_set.

    Args:
        term: The ontology term object to start from.
        lineage_set (set): A set to store the labels found along the path.
                           Should be initialized as set() for the first call.
    """
    # Use getattr for safer access to 'is_a'
    parents = getattr(term, 'is_a', [])

    for parent in parents:
        # Filter 1: Check if it's an ontology class
        if not isinstance(parent, owl.ThingClass):
            continue # Skip to the next parent in the list

        # Filter 2: Check if IRI contains relevant substrings (e.g., CL/GO)
        # Adjust IRI check based on your specific ontology needs
        parent_iri = str(parent.iri)
        if not ("CL_" in parent_iri or "GO_" in parent_iri or "/CL_" in parent_iri or "/GO_" in parent_iri):
             continue # Skip to the next parent in the list

        # --- First valid parent found ---

        # Get the parent label
        parent_label_list = getattr(parent, 'label', [])
        parent_label = parent_label_list.first() if parent_label_list else None

        # Process if label exists and is not already in the set
        if parent_label and parent_label not in lineage_set:
            lineage_set.append(parent_label)
            # Recurse using this specific parent
            find_single_path_lineage(parent, lineage_set)

        # Crucially, break the loop after processing the first valid parent.
        # Whether it was added or not, we stop looking at other parents
        # of the current 'term'.
        break

def find_lineage_with_steps(term, lineage_dict, current_depth=0):
    """
    Recursively finds 'is_a' parent labels, tracks their minimum distance
    (steps) from the initial term, and adds them to the lineage_dict.

    Args:
        term: The ontology term object to start from.
        lineage_dict (dict): A dictionary to store {parent_label: min_steps}.
                             Should be initialized as {} for the first call.
        current_depth (int): The current depth in the hierarchy (internal use).
    """
    # Calculate the depth for the direct parents of the current term
    parent_depth = current_depth + 1

    # Directly access 'is_a', assuming it exists
    # Use default=[] to handle cases where 'is_a' might be missing or None
    parents = getattr(term, 'is_a', [])

    for parent in parents:
        # --- Filtering (same as before) ---
        # Only interested in class hierarchy
        if not isinstance(parent, owl.ThingClass):
            continue

        # Filter for specific ontology prefixes if needed (optional)
        # Example: only CL or GO terms
        # Adjust IRI check based on your ontology structure
        parent_iri = str(parent.iri)
        if not ("CL_" in parent_iri or "GO_" in parent_iri or "/CL_" in parent_iri or "/GO_" in parent_iri): # Added checks for common IRI patterns
             continue


        # Directly get label, ensure it exists and is not empty
        parent_label_list = getattr(parent, 'label', [])
        if not parent_label_list:
            # Handle cases where label might be missing (e.g., use IRI fragment)
            # parent_label = parent.name
            continue # Or skip if label is essential
        parent_label = parent_label_list.first()
        if not parent_label:
             continue # Skip if label is empty

        # --- Logic for adding/updating parent and depth ---
        # Check if parent is already found and if the current path is shorter
        if parent_label not in lineage_dict or parent_depth < lineage_dict[parent_label]:
            # Add or update the parent with the new shorter depth
            lineage_dict[parent_label] = parent_depth
            # Recurse for this parent
            find_lineage_with_steps(parent, lineage_dict, parent_depth)
        # If parent_label is already in lineage_dict with a depth <= parent_depth,
        # we don't need to update or recurse further down this specific path,
        # as we've already found an equal or shorter path to it.

In [9]:


result = []

for _, row in pdf.iterrows():
    # find the term in the ontology
    cell_id = row['cell_type_ontology_term_id']
    search_term = cell_id.replace(":", "_")
    search_result = ontology.search_one(iri=f"*{search_term}")

    if not search_result is None:
        lineage = []
        find_single_path_lineage(search_result, lineage)
        cell_type = search_result.label[0]

        row = {
            'cell_id' : cell_id,
            'cell_type' : cell_type,
            'lineage' : lineage,
        }
        result.append(row)
        
    else:
        print(f"Skipping {cell_id}...")
        continue


result = pd.DataFrame(result)
print(f"raw: {result.shape=}")
result = result[result['cell_type'].notna()]
print(f"filtered: {result.shape=}")
# result = result.set_index(['cell_id', 'cell_type'])
result.head()

Skipping CL:4033026...
Skipping CL:4033043...
Skipping CL:4033042...
Skipping CL:4033039...
Skipping CL:4033038...
Skipping CL:4023072...
raw: result.shape=(183, 3)
filtered: result.shape=(183, 3)


,cell_id,cell_type,lineage
0,CL:0000057,fibroblast,"[connective tissue cell, animal cell, eukaryot..."
1,CL:0002553,fibroblast of lung,"[fibroblast, connective tissue cell, animal ce..."
2,CL:0002620,skin fibroblast,"[fibroblast, connective tissue cell, animal ce..."
3,CL:0000186,myofibroblast cell,"[contractile cell, native cell, cell]"
4,CL:0002241,pulmonary interstitial fibroblast,"[fibroblast, connective tissue cell, animal ce..."


In [10]:
# break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:

edge_counts = defaultdict(int)
for lineages in result['lineage']:
    for i in range(len(lineages) - 1):
        parent = lineages[i]
        child = lineages[i+1]
        edge_counts[(parent, child)] += 1

graph = nx.DiGraph()
for (parent, child), weight in edge_counts.items():
    graph.add_edge(parent, child, weight=weight)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 45, 20

# Use a hierarchical layout for the tree visualization
pos = nx.drawing.nx_agraph.graphviz_layout(graph, prog='dot', )

# Get edge weights for labels
edge_labels = {(u, v): d['weight'] for u, v, d in graph.edges(data=True)}


# selected nodes
selected_nodes = [
    # 'mature B cell',
    # 'professional antigen presenting cell',
    'leukocyte',
    'memory B cell',
    # 'antibody secreting cell',
    # 'lymphocyte of B lineage',
    'lymphocyte',
    # 'mononuclear cell',
    # 'single nucleate cell',
    # 'nucleate cell',
    # 'B cell, CD19-positive',
    'B cell',
    # 'mature natural killer cell',
    'natural killer cell',
    # 'group 1 innate lymphoid cell',
    'innate lymphoid cell',
    # 'CD8-positive, alpha-beta memory T cell, CD45RO-positive',
    # 'CD8-positive, alpha-beta memory T cell',
    # 'CD8-positive, alpha-beta T cell',
    'mature alpha-beta T cell',
    # 'alpha-beta T cell',
    'T cell',
    # 'lymphoid lineage restricted progenitor cell',
    'hematopoietic lineage restricted progenitor cell',
    'hematopoietic precursor cell',
    'hematopoietic cell',
    # 'animal cell',
    # 'eukaryotic cell',
    # 'CD4-positive, alpha-beta T cell',
    # 'pro-B cell',
    # 'pre-B-II cell',
    # 'precursor B cell',
    # 'CD4-positive, alpha-beta memory T cell, CD45RO-positive',
    # 'CD4-positive, alpha-beta memory T cell',
    # 'long lived plasma cell',
    'plasma cell',
    # 'activated CD4-positive, alpha-beta T cell',
    # 'CD4-positive helper T cell',
    # 'alpha-beta intraepithelial T cell',
    # 'B-2 B cell',
    # 'immature alpha-beta T cell',
    # 'germinal center B cell',
    'regulatory T cell',
    'mature T cell',
    # 'class switched memory B cell',
    # 'gamma-delta intraepithelial T cell',
    # 'mature gamma-delta T cell',
    'gamma-delta T cell',
    # 'activated CD8-positive, alpha-beta T cell',
    # 'thymocyte',
    # 'immature T cell',
    # 'effector T cell',
    # 'B-1 B cell',
    # 'CD4-positive, CD25-positive, alpha-beta regulatory T cell',
    # 'double negative thymocyte',
    # 'pro-T cell',
    # 'CD8-positive, alpha-beta cytokine secreting effector T cell',
    'erythroid lineage cell',
    'myeloid cell',
    'hematopoietic multipotent progenitor cell',
    'hematopoietic oligopotent progenitor cell',
    # 'somatic stem cell',
    # 'stem cell',
    # 'connective tissue cell',
    'fibroblast',
    # 'contractile cell',
    # 'pulmonary interstitial fibroblast',
    # 'fibroblast of lung',
    # 'cell of skeletal muscle',
    # 'somatic cell',
    'mesenchymal cell',
    # 'smooth muscle cell',
    # 'non-striated muscle cell',
    # 'muscle cell',
    # 'stuff accumulating cell',
    # 'multi fate stem cell',
    # 'squamous epithelial cell',
    'epithelial cell',
    # locstr('blood vessel smooth muscle cell', 'en'),
    # 'vascular associated smooth muscle cell',
    'pericyte',
    'monocyte',
    'myeloid leukocyte',
    'dendritic cell',
    'phagocyte',
    # 'tissue-resident macrophage',
    'macrophage',
    # 'elicited macrophage',
    'blood cell',
    # 'primitive erythroid lineage cell',
    'granulocyte',
    # 'myeloid dendritic cell',
    # 'immature innate lymphoid cell',
    # 'group 3 innate lymphoid cell',
    # 'group 3 innate lymphoid cell, human',
    # 'ILC1',
    # 'group 2 innate lymphoid cell',
    'blood vessel endothelial cell',
    # 'lining cell',
    # 'barrier cell',
    # 'capillary endothelial cell',
    # 'microvascular endothelial cell',
    # 'endothelial cell of hepatic sinusoid',
    # 'endothelial cell of sinusoid',
    'endothelial cell',
    'endothelial cell of vascular tree',
    # 'endothelial cell of artery',
    # 'lung endothelial cell',
    # 'epithelial cell of lung',
    # 'dermis lymphatic vessel endothelial cell',
    'endothelial cell of lymphatic vessel',
]



# Node colors: red for branching points, skyblue for others
node_colors = ['coral' if node in selected_nodes else 'skyblue' for node in graph.nodes()]

# Draw nodes
nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=2500)

# Draw edges with weights as labels
nx.draw_networkx_edges(graph, pos, width=1.5, alpha=0.6, edge_color='gray')

# Wrap node labels
wrapped_labels = {node: textwrap.fill(node, width=20) for node in graph.nodes()}

# Draw node labels
nx.draw_networkx_labels(graph, pos, labels=wrapped_labels, font_size=4, font_weight='bold')

plt.axis('off')
plt.show()

In [ ]:
def get_closest_selected_descendant(graph, node, selected_nodes):
    """
    Finds the closest descendant of a given node that is present in the
    'selected_nodes' set.

    Args:
        graph (nx.DiGraph): The directed graph representing the lineage.
        node (str): The node for which to find the closest selected descendant.
        selected_nodes (set): A set of selected node labels.

    Returns:
        str or None: The label of the closest selected descendant, or None if no
                     selected descendant is found.
    """
    if node in selected_nodes:
        return node

    closest_descendant = None
    distance = float('inf')
    queue = [(node, 0)]
    visited = {node}

    while queue:
        current_node, current_distance = queue.pop(0)
        for neighbor in graph.successors(current_node):
            if neighbor not in visited:
                visited.add(neighbor)
                if neighbor in selected_nodes:
                    if current_distance + 1 < distance:
                        distance = current_distance + 1
                        closest_descendant = neighbor
                queue.append((neighbor, current_distance + 1))

    return closest_descendant

test = 'pro-B cell'
get_closest_selected_descendant(graph, test, selected_nodes)

# Make the annotation file

In [ ]:
outpath = "../resources/cell_type_annotation.csv"
pdf = result.copy()

def get_row_annotation(x, i):
    try:
        return get_closest_selected_descendant(graph, x[i], selected_nodes)
    except:
        return None
        

for i in range(5):
    column_label = f"level_{i+1}"
    pdf[column_label] = pdf['lineage'].apply(lambda x: get_row_annotation(x, i))
    n_unique_values = pdf[column_label].nunique()
    print(f"{column_label} has {n_unique_values} unique values")

pdf.to_csv(outpath, index=False)
pdf.head()